# 02 Modelo de Fraude — A.U.R.A

Este notebook implementa el enfoque híbrido solicitado en el PDF:

- **Reglas de negocio (alertas explicables)** según señales del reto.
- **Modelo supervisado (ML)** usando `etiqueta_fraude_simulada`.
- **Detección de anomalías** (IsolationForest) para casos atípicos.
- **NLP** (capa aparte, PDF §9): similitud de narrativas → alimenta **reglas RF-07** (no entra al ML).
- **ML supervisado** (tabular): montos, fechas, documentos, proveedor, etc.
- **Anomalías**: IsolationForest.

> Importante: el resultado es un **score de riesgo** y recomendaciones de revisión; **no es una acusación** ni rechazo automático.


In [1]:
from pathlib import Path
import sys

# Permite importar `src.*` aunque ejecutes desde /notebooks
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.dataio.csv_exports import get_latest_export_dir, load_exported_tables

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest, HistGradientBoostingClassifier

EXPORT_DIR = get_latest_export_dir(Path("..") / "data" / "raw" / "supabase_export")
print(f"✅ Usando export CSV: {EXPORT_DIR}")


✅ Usando export CSV: ..\data\raw\supabase_export\20260528_094805


In [2]:
tables = load_exported_tables(EXPORT_DIR)

df_siniestros = tables["siniestros"]
df_polizas = tables["polizas"]
df_proveedores = tables["proveedores"]
df_documentos = tables["documentos"]

print('Siniestros:', len(df_siniestros))
print('Pólizas:', len(df_polizas))
print('Proveedores:', len(df_proveedores))
print('Documentos:', len(df_documentos))

# Fechas
for c in ['fecha_ocurrencia','fecha_reporte']:
    if c in df_siniestros.columns:
        df_siniestros[c] = pd.to_datetime(df_siniestros[c], errors='coerce')
for c in ['fecha_inicio_vigencia','fecha_fin_vigencia']:
    if c in df_polizas.columns:
        df_polizas[c] = pd.to_datetime(df_polizas[c], errors='coerce')


Siniestros: 1000
Pólizas: 1000
Proveedores: 50
Documentos: 1993


In [3]:
# Dataset maestro (siniestros + pólizas + proveedor agregados)

df = df_siniestros.merge(
    df_polizas[['id_poliza','monto_asegurado','tipo_cobertura','ramo','ciudad','estado_poliza','deducible','prima']]
        if 'id_poliza' in df_polizas.columns else df_polizas,
    on='id_poliza',
    how='left',
    suffixes=('', '_pol')
)

df = df.merge(
    df_proveedores[['id_proveedor','tipo_proveedor','ciudad','lista_restrictiva','proveedor_preferente']]
        if 'id_proveedor' in df_proveedores.columns else df_proveedores,
    on='id_proveedor',
    how='left',
    suffixes=('', '_prov')
)

# Flags agregados desde documentos
if not df_documentos.empty and 'id_siniestro' in df_documentos.columns:
    doc = df_documentos.copy()
    # asegurar columnas
    for c in ['posible_adulteracion','inconsistencia_detectada','es_legible','entregado']:
        if c not in doc.columns:
            doc[c] = False

    # Agregaciones por siniestro (compatible con pandas NamedAgg)
    agg = doc.groupby('id_siniestro').agg(
        doc_count=('id_documento', 'count'),
        doc_inconsistencia=('inconsistencia_detectada', 'max'),
        doc_adulteracion=('posible_adulteracion', 'max'),
        doc_entregado=('entregado', 'min'),
        doc_es_legible=('es_legible', 'min'),
    )

    # Flags derivados
    agg['doc_ilegible'] = (~agg['doc_es_legible'].astype(bool)).astype(int)
    agg['doc_faltante'] = (~agg['doc_entregado'].astype(bool)).astype(int)

    df = df.merge(agg.reset_index(), on='id_siniestro', how='left')

# Features derivadas
if {'monto_reclamado','monto_asegurado'}.issubset(df.columns):
    df['ratio_reclamado'] = df['monto_reclamado'] / df['monto_asegurado']

df['delta_notificacion_dias'] = df.get('dias_entre_ocurrencia_reporte', np.nan)
df['delta_vigencia_dias'] = df.get('dias_desde_inicio_poliza', np.nan)

# Limpieza
for c in ['doc_count','doc_inconsistencia','doc_adulteracion','doc_ilegible','doc_faltante']:
    if c in df.columns:
        df[c] = df[c].fillna(0)

print('✅ df master:', df.shape)
display(df[['codigo_siniestro','cobertura','monto_reclamado','ratio_reclamado','delta_notificacion_dias','delta_vigencia_dias','documentos_completos','etiqueta_fraude_simulada']].head())


✅ df master: (1000, 53)


,codigo_siniestro,cobertura,monto_reclamado,ratio_reclamado,delta_notificacion_dias,delta_vigencia_dias,documentos_completos,etiqueta_fraude_simulada
0,SIN-2026-0001,ROBO,2762.36,0.142581,1,7,True,1
1,SIN-2026-0002,ROBO,2959.25,0.136296,2,55,True,0
2,SIN-2026-0003,CHOQUE,594.41,0.032016,1,134,True,0
3,SIN-2026-0004,CHOQUE,3819.04,0.205126,2,97,True,0
4,SIN-2026-0005,ROBO,2218.25,0.064232,2,161,True,0


In [4]:
# =========================
# 1) Capa NLP → Reglas (PDF §9: análisis textual separado del ML)
# =========================

from src.nlp.narrative_similarity import add_narrative_similarity_features
from src.rules.fraud_rules import calcular_score_reglas

# NLP solo para reglas (RF-07). El ML NO usa max_similitud_narrativa.
df = add_narrative_similarity_features(df)
print('✅ Capa NLP: max_similitud_narrativa (para reglas RF-07)')

rules = df.apply(calcular_score_reglas, axis=1)
df['score_reglas'] = [x[0] for x in rules]
df['razones_reglas'] = [x[1] for x in rules]

print('✅ Capa reglas listo. Resumen:')
display(df['score_reglas'].describe())


✅ Capa NLP: max_similitud_narrativa (para reglas RF-07)
✅ Capa reglas listo. Resumen:


count    1000.000000
mean       14.242000
std         5.732041
min         8.000000
25%        11.000000
50%        12.000000
75%        18.000000
max        43.000000
Name: score_reglas, dtype: float64

In [5]:
# =========================
# 2) Modelo supervisado (ML)
# =========================

# Variables
label = 'etiqueta_fraude_simulada'

# ML tabular (sin NLP — ver capa 1)
feature_cols_num = [
    'monto_reclamado','monto_estimado','ratio_reclamado',
    'delta_notificacion_dias','delta_vigencia_dias',
    'historial_siniestros_asegurado',
    'doc_count','doc_inconsistencia','doc_adulteracion','doc_ilegible','doc_faltante',
]
feature_cols_cat = ['cobertura','sucursal','tipo_proveedor']
feature_cols_bool = ['documentos_completos','lista_restrictiva','proveedor_preferente']

for c in feature_cols_num:
    if c not in df.columns:
        df[c] = np.nan
for c in feature_cols_cat:
    if c not in df.columns:
        df[c] = None
for c in feature_cols_bool:
    if c not in df.columns:
        df[c] = False

X = df[feature_cols_num + feature_cols_cat + feature_cols_bool].copy()
# sklearn SimpleImputer (según versión) puede fallar con dtype bool -> convertir a 0/1
for c in feature_cols_bool:
    X[c] = X[c].fillna(False).astype(int)

y = df[label].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pre = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imp', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), feature_cols_num),
        ('cat', Pipeline(steps=[('imp', SimpleImputer(strategy='most_frequent')),
                               ('oh', OneHotEncoder(handle_unknown='ignore'))]), feature_cols_cat),
        # bool ya viene como 0/1 (int) en X
        ('bool', Pipeline(steps=[('imp', SimpleImputer(strategy='most_frequent'))]), feature_cols_bool),
    ]
)

clf_lr = Pipeline(steps=[
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

clf_rf = Pipeline(steps=[
    ('pre', pre),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'))
])

# Boosting (buen rendimiento en tabular)
clf_hgb = Pipeline(steps=[
    ('pre', pre),
    ('clf', HistGradientBoostingClassifier(random_state=42))
])

models = {
    'LogisticRegression': clf_lr,
    'RandomForest': clf_rf,
    'HistGradientBoosting': clf_hgb,
}

rows = []
for name, model in models.items():
    print(f'Entrenando {name}...')
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, proba)
    rep = classification_report(y_test, pred, output_dict=True, zero_division=0)
    rows.append({
        'model': name,
        'auc': auc,
        'f1_fraude': rep['1']['f1-score'],
        'recall_fraude': rep['1']['recall'],
        'precision_fraude': rep['1']['precision'],
    })

results = pd.DataFrame(rows).sort_values('auc', ascending=False)
display(results)

# Elegimos el mejor por AUC
best_name = results.iloc[0]['model']
best = models[best_name]
proba_best = best.predict_proba(X_test)[:, 1]

print(f'\n✅ Mejor modelo por AUC: {best_name}')
print('Reporte (best, threshold=0.5):')
print(classification_report(y_test, (proba_best>=0.5).astype(int), zero_division=0))
print('Matriz confusión:\n', confusion_matrix(y_test, (proba_best>=0.5).astype(int)))


Entrenando LogisticRegression...
Entrenando RandomForest...
Entrenando HistGradientBoosting...


,model,auc,f1_fraude,recall_fraude,precision_fraude
0,LogisticRegression,0.930556,0.651685,0.852941,0.527273
1,RandomForest,0.901620,0.785714,0.647059,1.000000
2,HistGradientBoosting,0.848175,0.766667,0.676471,0.884615



✅ Mejor modelo por AUC: LogisticRegression
Reporte (best, threshold=0.5):
              precision    recall  f1-score   support

           0       0.97      0.88      0.92       216
           1       0.53      0.85      0.65        34

    accuracy                           0.88       250
   macro avg       0.75      0.87      0.79       250
weighted avg       0.91      0.88      0.89       250

Matriz confusión:
 [[190  26]
 [  5  29]]


In [6]:
# =========================
# 3) Modelo de anomalías
# =========================

# Usamos solo numéricas (más estable)
X_num = df[feature_cols_num].copy()
X_num = X_num.fillna(X_num.median(numeric_only=True))

iso = IsolationForest(n_estimators=300, random_state=42, contamination=0.10)
iso.fit(X_num)

# IsolationForest: score_samples -> mayor es más normal. Invertimos para riesgo.
raw = -iso.score_samples(X_num)
raw_norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)

df['anom_score_0_1'] = raw_norm
print('✅ Anomalías listo:', df['anom_score_0_1'].describe())


✅ Anomalías listo: count    1000.000000
mean        0.264978
std         0.193323
min         0.000000
25%         0.127471
50%         0.203529
75%         0.328938
max         1.000000
Name: anom_score_0_1, dtype: float64


In [7]:
# =========================
# 4) Score final 0–100 + semáforo + explicación
# =========================

# Probabilidad ML (RF) para todos
proba_all = best.predict_proba(X)[:,1]
df['ml_proba'] = proba_all

# Pesos (alineados al PDF: reglas + ML + anomalías; NLP opcional)
W_RULES = 70
W_ML = 20
W_ANOM = 10

# score final
score_final = (
    (df['score_reglas'].clip(0, 70) / 70.0) * W_RULES
    + df['ml_proba'].clip(0,1) * W_ML
    + df['anom_score_0_1'].clip(0,1) * W_ANOM
)

df['score_final'] = score_final.round(0).astype(int).clip(0, 100)

from src.rules.fraud_rules import calcular_semaforo

df['semaforo'] = df['score_final'].apply(calcular_semaforo)

# Explicación simple

def explicar(row: pd.Series) -> str:
    razones = list(row.get('razones_reglas') or [])
    # Top contribuciones por módulos
    razones.append(f"ML prob={row.get('ml_proba',0):.2f}")
    razones.append(f"Anomalía={row.get('anom_score_0_1',0):.2f}")
    return ' | '.join(razones[:6])

df['explicacion'] = df.apply(explicar, axis=1)

print('Distribución semáforo:')
display(df['semaforo'].value_counts())

print('Top 10 siniestros por score (para demo):')
display(df.sort_values('score_final', ascending=False)[['codigo_siniestro','score_final','semaforo','cobertura','sucursal','id_proveedor','explicacion']].head(10))


Distribución semáforo:


semaforo
VERDE       914
AMARILLO     86
Name: count, dtype: int64

Top 10 siniestros por score (para demo):


,codigo_siniestro,score_final,semaforo,cobertura,sucursal,id_proveedor,explicacion
171,SIN-2026-0172,67,AMARILLO,CHOQUE,MANTA,prov-0776898a,Borde de vigencia 11–30d (+4) | Reporte tardío...
622,SIN-2026-0623,65,AMARILLO,ROBO,GUAYAQUIL,prov-b0d3cc8d,Borde de vigencia <=10d (+8) | RF-06: Robo con...
384,SIN-2026-0385,62,AMARILLO,ROBO,CUENCA,prov-b6b79676,RF-06: Robo con reporte tardío >4d (+8) | RF-0...
369,SIN-2026-0370,58,AMARILLO,DAÑOS,CUENCA,prov-7d13740f,RF-05: Borde de vigencia <48h (+10) | Reporte ...
560,SIN-2026-0561,57,AMARILLO,ROBO,MANTA,prov-b6b79676,Borde de vigencia <=10d (+8) | RF-06: Robo con...
147,SIN-2026-0148,57,AMARILLO,ROBO,MANTA,prov-7d13740f,RF-05: Borde de vigencia <48h (+10) | Reporte ...
632,SIN-2026-0633,57,AMARILLO,ROBO,MANTA,prov-c70782ce,Reporte tardío 4–7d (+3) | RF-02: Evidencia do...
837,SIN-2026-0838,56,AMARILLO,CHOQUE,QUITO,prov-0776898a,Reporte tardío 4–7d (+3) | RF-03: Proveedor en...
203,SIN-2026-0204,56,AMARILLO,ROBO,QUITO,prov-0776898a,RF-06: Robo con reporte tardío >4d (+8) | RF-0...
554,SIN-2026-0555,56,AMARILLO,DAÑOS,GUAYAQUIL,prov-b6b79676,Borde de vigencia <=10d (+8) | Reporte tardío ...
